In [ ]:
import torch
import matplotlib.pyplot as plt


def laplace_jacobi_1d_step(v: torch.Tensor, omega: float = 2/3) -> torch.Tensor:
    """
    Runs damped Jacobi for the 1D Laplace equation (Au = 0).
    Assumes homogeneous Dirichlet boundary conditions: v[0] = v[-1] = 0.

    Args:
        v: 1D torch.Tensor representing the current solution vector.
        omega: The damping factor (default 2/3 for optimal high-frequency damping).
    """
    v_out = v.clone()

    # 1D Laplace Jacobi step: 0.5 * (left_neighbor + right_neighbor)
    v_jacobi = 0.5 * (v_out[:-2] + v_out[2:])

    # Damped update for the interior points
    v_out[1:-1] = (1 - omega) * v_out[1:-1] + omega * v_jacobi


def laplace_jacobi_1d_smoothen(v: torch.Tensor, omega: float = 2/3, iterations: int = 10) -> torch.Tensor:
    """
    Runs damped Jacobi for the 1D Laplace equation (Au = 0).
    Assumes homogeneous Dirichlet boundary conditions: v[0] = v[-1] = 0.

    Args:
        v: 1D torch.Tensor representing the initial guess.
        omega: The damping factor (default 2/3 for optimal high-frequency damping).
        iterations: Number of Jacobi steps to perform.
    """

    for _ in range(iterations):
        v = laplace_jacobi_1d_step(v, omega)

    return v



# 1. Setup the grid (n=64 grid intervals)
n = 64
x = torch.linspace(0, n, n+1)

# 2. Specify the initial guess consisting of all modes
v_initial = torch.zeros(n+1)
for k in range(1, n):
    v_initial = v_initial + torch.sin(k * torch.pi * x / n)

# Enforce v_0 = v_n = 0
v_initial[0] = 0.0
v_initial[-1] = 0.0


# ================================
# run Jacobi and Multigrid V-Cycle
# ================================
total_iterations = 50

# run Jacobi
jacobi_norms = torch.empty(total_iterations)

v = v_initial.clone()
for i in range(total_iterations):
    v = laplace_jacobi_1d_step(v, omega=2/3)
    jacobi_norms[i] = v.norm()


#run Multigrid V-Cycle




